# LeetCode 244 — Shortest Word Distance II bằng Python

Link đề: https://leetcode.com/problems/shortest-word-distance-ii/ · Độ khó: **Medium** · Chủ đề: Design, Array, Hash Table, Two Pointers, String

Skeleton LeetCode:

```python
class WordDistance:

    def __init__(self, wordsDict: List[str]):
        ...

    def shortest(self, word1: str, word2: str) -> int:
        ...


# Your WordDistance object will be instantiated and called as such:
# obj = WordDistance(wordsDict)
# param_1 = obj.shortest(word1,word2)
```

Ghi chú: đây là bài **Premium** của LeetCode; đề bài viết theo hiểu biết về bài (và bản mô tả đã tải), nên đối chiếu lại nếu có tài khoản Premium. Mọi đáp án trong notebook đều được kiểm chứng bằng code, kể cả bằng một lời giải độc lập duyệt mọi cặp chỉ số.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề **thiết kế cấu trúc dữ liệu**: lớp `WordDistance` được khởi tạo **một lần** với mảng chuỗi `wordsDict`, sau đó trả lời **nhiều truy vấn** `shortest(word1, word2)` = khoảng cách ngắn nhất (hiệu hai chỉ số) giữa một lần xuất hiện của `word1` và một lần xuất hiện của `word2`.
2. Biết cách **viết một lớp trong Python** (`__init__`, thuộc tính đối tượng, phương thức) và cách **mô phỏng định dạng test của bài thiết kế** (danh sách tên lệnh + danh sách tham số).
3. Nhận ra **ý tưởng then chốt**: truy vấn nhiều lần nên **tiền xử lý**: với mỗi từ lưu danh sách **chỉ số xuất hiện (đã sắp tăng)**; mỗi truy vấn chỉ làm việc với hai danh sách nhỏ này.
4. Đi từ **quét cả mảng cho mỗi truy vấn `O(n)`** → **duyệt mọi cặp chỉ số `O(a·b)`** → **hai con trỏ trộn hai danh sách đã sắp `O(a + b)`** (cách nộp bài) → **tìm nhị phân `O(a log b)`** → **thêm bộ nhớ đệm kết quả truy vấn**.
5. Tránh các lỗi kinh điển: chỉ lưu **một** chỉ số cho mỗi từ (ghi đè), tăng sai con trỏ, quên `abs`, và **dùng thuộc tính của lớp thay vì của đối tượng** (nhiều đối tượng dùng chung dữ liệu).

## 2. Đề bài & ví dụ

Thiết kế một cấu trúc dữ liệu được khởi tạo với một mảng chuỗi, rồi trả lời các truy vấn về **khoảng cách ngắn nhất giữa hai chuỗi khác nhau** trong mảng.

Cài đặt lớp `WordDistance`:

- `WordDistance(String[] wordsDict)`: khởi tạo đối tượng với mảng `wordsDict`.
- `int shortest(String word1, String word2)`: trả về khoảng cách ngắn nhất giữa `word1` và `word2` trong `wordsDict` (khoảng cách giữa hai lần xuất hiện ở chỉ số `i` và `j` là `|i − j|`; lấy nhỏ nhất trên mọi cặp lần xuất hiện).

**Ví dụ 1**

```
Input
["WordDistance", "shortest", "shortest"]
[[["practice", "makes", "perfect", "coding", "makes"]], ["coding", "practice"], ["makes", "coding"]]
Output
[null, 3, 1]
```

Giải thích: `wordsDict = ["practice", "makes", "perfect", "coding", "makes"]`; `shortest("coding", "practice")` trả `3` (chỉ số `3` và `0`); `shortest("makes", "coding")` trả `1` (chỉ số `4` và `3`).

**Ràng buộc:** `1 <= wordsDict.length <= 3 * 10⁴`; `1 <= wordsDict[i].length <= 10`; `wordsDict[i]` gồm chữ cái thường; `word1` và `word2` có trong `wordsDict`; `word1 != word2`; có tối đa `5000` lần gọi `shortest`.

In [1]:
from typing import List
import random
from math import inf
from bisect import bisect_left
from collections import defaultdict

WORDS = ["practice", "makes", "perfect", "coding", "makes"]
examples = [
    (WORDS, [("coding", "practice", 3), ("makes", "coding", 1)]),
]
print("wordsDict =", WORDS)
for words, queries in examples:
    for w1, w2, expected in queries:
        print(f"shortest({w1!r}, {w2!r}) kỳ vọng {expected}")

wordsDict = ['practice', 'makes', 'perfect', 'coding', 'makes']
shortest('coding', 'practice') kỳ vọng 3
shortest('makes', 'coding') kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Lớp (class), `__init__` và thuộc tính đối tượng

Bài thiết kế yêu cầu viết một **lớp**. `__init__(self, ...)` chạy khi tạo đối tượng và thường lưu dữ liệu vào **thuộc tính đối tượng** `self.xxx`; các phương thức khác (`shortest`) dùng lại dữ liệu đó. Mỗi đối tượng có bộ thuộc tính **riêng**.

In [2]:
class Counter2:
    def __init__(self, start):
        self.value = start            # thuộc tính của ĐỐI TƯỢNG

    def add(self, x):
        self.value += x
        return self.value


a, b = Counter2(0), Counter2(100)
print(a.add(5), a.add(5), b.add(1))    # 5 10 101: hai đối tượng không ảnh hưởng nhau

5 10 101


### 3.2. `dict` các danh sách: `defaultdict(list)` và `enumerate`

Để biết một từ xuất hiện ở những chỉ số nào ta dùng `dict` ánh xạ `từ → list chỉ số`. `defaultdict(list)` tự tạo list rỗng cho khóa mới. Duyệt `for i, w in enumerate(words)` theo `i` **tăng dần** nên mỗi list tự động **đã sắp tăng**.

In [3]:
positions = defaultdict(list)
for i, w in enumerate(WORDS):
    positions[w].append(i)
print(dict(positions))
# {'practice': [0], 'makes': [1, 4], 'perfect': [2], 'coding': [3]}
print(positions["makes"], positions["coding"])

{'practice': [0], 'makes': [1, 4], 'perfect': [2], 'coding': [3]}
[1, 4] [3]


### 3.3. Khoảng cách giữa hai chỉ số và giá trị tuyệt đối

Khoảng cách giữa hai lần xuất hiện ở chỉ số `i` và `j` là `|i − j| = abs(i - j)`. Không có `abs` thì có thể ra số âm (khi `word1` đứng sau `word2`).

In [4]:
print(abs(3 - 0), abs(0 - 3), 0 - 3)            # 3 3 -3
a, b = [1, 4], [3]                              # chỉ số của "makes" và "coding"
print([abs(i - j) for i in a for j in b])       # [2, 1]: nhỏ nhất là 1

3 3 -3
[2, 1]


### 3.4. Hai con trỏ trên hai danh sách đã sắp (trộn kiểu merge)

Cho hai list `a`, `b` đã sắp tăng. Đặt `i = j = 0`; so `a[i]` với `b[j]`: cập nhật đáp án bằng `abs(a[i] − b[j])`, rồi **tăng con trỏ của phần tử nhỏ hơn**. Lý do: phần tử nhỏ hơn không thể gần hơn với bất kỳ phần tử sau nào của list kia (chúng còn lớn hơn). Mỗi bước loại một phần tử, nên tổng `≤ len(a) + len(b)` bước.

In [5]:
def merge_min_gap(a, b):
    i = j = 0
    best = inf
    while i < len(a) and j < len(b):
        best = min(best, abs(a[i] - b[j]))
        if a[i] <= b[j]:
            i += 1                 # a[i] là nhỏ hơn: không thể gần hơn với b[j+1:] (lớn hơn b[j])
        else:
            j += 1
    return best


print(merge_min_gap([1, 4], [3]))          # 1
print(merge_min_gap([0, 10], [5, 7, 20]))  # 3  (7 và 10)

1
3


### 3.5. Tìm nhị phân với `bisect_left`

Trong list đã sắp `b`, `bisect_left(b, x)` cho vị trí chèn `x`; hai ứng viên gần `x` nhất là `b[pos − 1]` và `b[pos]` (nếu tồn tại). Đây là cách tìm phần tử gần nhất trong `O(log len(b))`.

In [6]:
b = [3, 8, 15]
for x in [1, 5, 9, 20]:
    pos = bisect_left(b, x)
    cands = [b[k] for k in (pos - 1, pos) if 0 <= k < len(b)]
    print(f"x={x}: pos={pos}, ứng viên {cands}, gần nhất cách {min(abs(x - c) for c in cands)}")

x=1: pos=0, ứng viên [3], gần nhất cách 2
x=5: pos=1, ứng viên [3, 8], gần nhất cách 2
x=9: pos=2, ứng viên [8, 15], gần nhất cách 1
x=20: pos=3, ứng viên [15], gần nhất cách 5


### 3.6. Mô phỏng định dạng test của bài thiết kế

LeetCode cho bài thiết kế dưới dạng hai danh sách: `ops` (tên lệnh, phần tử đầu là tên lớp) và `args` (danh sách tham số cho từng lệnh). Ta viết `run_design(cls, ops, args)` tạo đối tượng rồi gọi lần lượt các phương thức; kết quả có `None` ở vị trí hàm tạo.

In [7]:
def run_design(cls, ops, args):
    obj = None
    out = []
    for op, arg in zip(ops, args):
        if op == cls.__name__:
            obj = cls(*arg)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out


class Demo:
    def __init__(self, words):
        self.words = words

    def count(self, w):
        return self.words.count(w)


print(run_design(Demo, ["Demo", "count", "count"], [[["a", "b", "a"]], ["a"], ["c"]]))   # [None, 2, 0]

[None, 2, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`wordsDict = ["practice", "makes", "perfect", "coding", "makes"]` (chỉ số `0..4`).

Truy vấn `shortest("makes", "coding")`:

- `"makes"` xuất hiện ở chỉ số `1` và `4`;
- `"coding"` xuất hiện ở chỉ số `3`.

Các khoảng cách `|1 − 3| = 2`, `|4 − 3| = 1`. Nhỏ nhất là `1`.

Điều đáng chú ý: để trả lời, ta chỉ cần **các chỉ số xuất hiện của hai từ**, không cần đọc cả mảng. Nếu có nhiều truy vấn, **gom sẵn** các chỉ số của mỗi từ một lần ngay từ đầu (ở hàm khởi tạo) sẽ tiết kiệm công sức rất nhiều: mỗi truy vấn chỉ dùng hai danh sách nhỏ.

In [8]:
words = WORDS
a = [i for i, w in enumerate(words) if w == "makes"]
b = [i for i, w in enumerate(words) if w == "coding"]
print("makes ở", a, "| coding ở", b)
print("mọi khoảng cách:", sorted(abs(i - j) for i in a for j in b), "-> nhỏ nhất", min(abs(i - j) for i in a for j in b))

makes ở [1, 4] | coding ở [3]
mọi khoảng cách: [1, 2] -> nhỏ nhất 1


## 5. Cách 1 — Brute Force / baseline: quét cả mảng cho mỗi truy vấn

**Ý tưởng.** Khởi tạo chỉ **lưu mảng**. Mỗi truy vấn quét mảng một lần, ghi nhớ chỉ số gần nhất của `word1` và `word2` đã gặp và cập nhật đáp án mỗi khi gặp một trong hai từ (đây chính là lời giải bài "Shortest Word Distance I").

In [9]:
class WordDistanceScan:
    def __init__(self, wordsDict: list[str]):
        self.words = wordsDict

    def shortest(self, word1: str, word2: str) -> int:
        i1 = i2 = -1                                   # chỉ số gần nhất đã gặp của mỗi từ
        best = len(self.words)
        for k, w in enumerate(self.words):             # O(n) mỗi truy vấn
            if w == word1:
                i1 = k
            elif w == word2:
                i2 = k
            if i1 != -1 and i2 != -1:
                best = min(best, abs(i1 - i2))
        return best


for words, queries in examples:
    d = WordDistanceScan(words)
    for w1, w2, expected in queries:
        print(f"shortest({w1!r}, {w2!r}) = {d.shortest(w1, w2)} (kỳ vọng {expected})")

shortest('coding', 'practice') = 3 (kỳ vọng 3)
shortest('makes', 'coding') = 1 (kỳ vọng 1)


**Độ phức tạp (Cách 1).** Gọi `n = len(wordsDict)` và `Q` là số truy vấn. Khởi tạo `O(1)`; mỗi truy vấn `O(n)`; tổng **`O(Q · n)`**. Với `n = 3·10⁴` và `Q = 5000` là `1.5·10⁸` phép so sánh chuỗi: **chậm**. **Space `O(1)`** thêm.

**Nút thắt.** Mỗi truy vấn đọc lại **cả** mảng dù chỉ hai từ quan trọng. Nếu biết trước vị trí các từ, mỗi truy vấn chỉ cần xét vài vị trí.

## 6. Ý tưởng thuật toán / bước đột phá

**Tiền xử lý một lần, truy vấn nhiều lần.** Đề báo trước sẽ có nhiều truy vấn trên **cùng** một mảng, nên ta bỏ công sức ở hàm khởi tạo:

```
positions[w] = [các chỉ số i sao cho wordsDict[i] == w]   (tăng dần)
```

Tạo `positions` mất `O(n)` một lần. Mỗi truy vấn `shortest(word1, word2)`:

- lấy `a = positions[word1]`, `b = positions[word2]` (đã sắp tăng);
- tìm `min |x − y|` với `x ∈ a`, `y ∈ b` bằng **hai con trỏ** (trộn kiểu merge) trong `O(len(a) + len(b))`.

Tổng độ dài các danh sách của mọi từ là `n`, nên một truy vấn tốn nhiều nhất `O(n)` nhưng thường rất nhỏ; trường hợp xấu (hai từ chiếm gần hết mảng) mới tiệm cận `O(n)`.

**Vì sao hai con trỏ đúng?** Xét `a[i]` và `b[j]`: nếu `a[i] ≤ b[j]` thì mọi `b[j'] ≥ b[j]` với `j' > j` đều xa `a[i]` hơn `b[j]` (hoặc bằng), nên ta đã xét xong `a[i]` và tăng `i`. Đối xứng khi `a[i] > b[j]`.

In [10]:
positions = defaultdict(list)
for i, w in enumerate(WORDS):
    positions[w].append(i)
a, b = positions["makes"], positions["coding"]
i = j = 0
best = inf
while i < len(a) and j < len(b):
    print(f"i={i}, j={j}: a[i]={a[i]}, b[j]={b[j]}, |hiệu|={abs(a[i] - b[j])}")
    best = min(best, abs(a[i] - b[j]))
    if a[i] <= b[j]:
        i += 1
    else:
        j += 1
print("khoảng cách ngắn nhất:", best)

i=0, j=0: a[i]=1, b[j]=3, |hiệu|=2
i=1, j=0: a[i]=4, b[j]=3, |hiệu|=1
khoảng cách ngắn nhất: 1


## 7. Các cách tiếp theo

### Cách 2 — Lưu danh sách chỉ số, mỗi truy vấn duyệt mọi cặp

**Giải thích.** Tiền xử lý `positions` như mục 6; mỗi truy vấn tính `min(abs(x − y))` trên **mọi cặp** `(x, y)` với `x ∈ a`, `y ∈ b`. Đơn giản nhưng tốn `O(len(a) · len(b))`: tệ khi hai từ đều xuất hiện nhiều lần.

In [11]:
class WordDistancePairs:
    def __init__(self, wordsDict: list[str]):
        self.pos = defaultdict(list)
        for i, w in enumerate(wordsDict):
            self.pos[w].append(i)

    def shortest(self, word1: str, word2: str) -> int:
        a, b = self.pos[word1], self.pos[word2]
        return min(abs(x - y) for x in a for y in b)      # mọi cặp


for words, queries in examples:
    d = WordDistancePairs(words)
    for w1, w2, expected in queries:
        print(f"shortest({w1!r}, {w2!r}) = {d.shortest(w1, w2)} (kỳ vọng {expected})")

shortest('coding', 'practice') = 3 (kỳ vọng 3)
shortest('makes', 'coding') = 1 (kỳ vọng 1)


**Độ phức tạp Cách 2:** khởi tạo **`O(n)`**; mỗi truy vấn **`O(a·b)`** với `a`, `b` là số lần xuất hiện của hai từ (tệ nhất `O(n²/4)` nếu mỗi từ chiếm nửa mảng). **Space `O(n)`**.

### Cách 3 — Hai con trỏ trên hai danh sách đã sắp (cách nộp bài)

**Giải thích.** Dùng thuật toán trộn ở mục 6 cho mỗi truy vấn.

In [12]:
class WordDistanceTwoPointers:
    def __init__(self, wordsDict: list[str]):
        self.pos = defaultdict(list)
        for i, w in enumerate(wordsDict):
            self.pos[w].append(i)                       # tăng dần theo i

    def shortest(self, word1: str, word2: str) -> int:
        a, b = self.pos[word1], self.pos[word2]
        i = j = 0
        best = inf
        while i < len(a) and j < len(b):
            best = min(best, abs(a[i] - b[j]))
            if a[i] <= b[j]:
                i += 1
            else:
                j += 1
        return best


for words, queries in examples:
    d = WordDistanceTwoPointers(words)
    for w1, w2, expected in queries:
        print(f"shortest({w1!r}, {w2!r}) = {d.shortest(w1, w2)} (kỳ vọng {expected})")

shortest('coding', 'practice') = 3 (kỳ vọng 3)
shortest('makes', 'coding') = 1 (kỳ vọng 1)


In [13]:
# Code nộp LeetCode cho Cách 3
from collections import defaultdict
from math import inf
from typing import List


class WordDistance:
    def __init__(self, wordsDict: List[str]):
        self.d = defaultdict(list)
        for i, w in enumerate(wordsDict):
            self.d[w].append(i)

    def shortest(self, word1: str, word2: str) -> int:
        a, b = self.d[word1], self.d[word2]
        ans = inf
        i = j = 0
        while i < len(a) and j < len(b):
            ans = min(ans, abs(a[i] - b[j]))
            if a[i] <= b[j]:
                i += 1
            else:
                j += 1
        return ans

**Độ phức tạp Cách 3:** khởi tạo **`O(n)`**, mỗi truy vấn **`O(a + b)`**, **Space `O(n)`** cho `positions`.

### Cách 4 — Tìm nhị phân: với mỗi chỉ số của list ngắn hơn, tìm chỉ số gần nhất trong list dài hơn

**Giải thích.** Gọi `short` là list ngắn hơn, `long` là list dài hơn (cả hai đã sắp). Với mỗi `x` trong `short`, dùng `bisect_left(long, x)` tìm vị trí chèn, rồi xét hai phần tử lân cận trong `long`. Tốn `O(len(short) · log len(long))`, rất tốt khi một từ **hiếm** còn từ kia **phổ biến**.

In [14]:
class WordDistanceBisect:
    def __init__(self, wordsDict: list[str]):
        self.pos = defaultdict(list)
        for i, w in enumerate(wordsDict):
            self.pos[w].append(i)

    def shortest(self, word1: str, word2: str) -> int:
        a, b = self.pos[word1], self.pos[word2]
        short, long_ = (a, b) if len(a) <= len(b) else (b, a)
        best = inf
        for x in short:
            k = bisect_left(long_, x)                   # vị trí chèn x vào danh sách dài
            if k < len(long_):
                best = min(best, long_[k] - x)          # phần tử >= x gần nhất
            if k > 0:
                best = min(best, x - long_[k - 1])      # phần tử < x gần nhất
        return best


for words, queries in examples:
    d = WordDistanceBisect(words)
    for w1, w2, expected in queries:
        print(f"shortest({w1!r}, {w2!r}) = {d.shortest(w1, w2)} (kỳ vọng {expected})")

shortest('coding', 'practice') = 3 (kỳ vọng 3)
shortest('makes', 'coding') = 1 (kỳ vọng 1)


**Độ phức tạp Cách 4:** khởi tạo `O(n)`, mỗi truy vấn **`O(min(a, b) · log max(a, b))`**; **Space `O(n)`**. Hơn Cách 3 khi một danh sách ngắn hơn nhiều; kém hơn khi hai danh sách cùng cỡ (`a log a` so với `2a`).

### Cách 5 — Hai con trỏ kèm bộ nhớ đệm kết quả truy vấn

**Giải thích.** Nếu cùng một cặp từ được hỏi **lặp lại**, ta lưu kết quả trong `dict` theo khóa là cặp từ **không phụ thuộc thứ tự** (`frozenset` hoặc tuple đã sắp), và trả ngay ở những lần sau. Khoảng cách đối xứng (`shortest(a, b) == shortest(b, a)`).

In [15]:
class WordDistanceCached:
    def __init__(self, wordsDict: list[str]):
        self.pos = defaultdict(list)
        for i, w in enumerate(wordsDict):
            self.pos[w].append(i)
        self.cache = {}                                  # (từ nhỏ hơn, từ lớn hơn) -> kết quả

    def shortest(self, word1: str, word2: str) -> int:
        key = (word1, word2) if word1 <= word2 else (word2, word1)      # đối xứng
        if key in self.cache:
            return self.cache[key]
        a, b = self.pos[word1], self.pos[word2]
        i = j = 0
        best = inf
        while i < len(a) and j < len(b):
            best = min(best, abs(a[i] - b[j]))
            if a[i] <= b[j]:
                i += 1
            else:
                j += 1
        self.cache[key] = best
        return best


for words, queries in examples:
    d = WordDistanceCached(words)
    for w1, w2, expected in queries:
        print(f"shortest({w1!r}, {w2!r}) = {d.shortest(w1, w2)} (kỳ vọng {expected})")
    print("hỏi lại (lần 2, lấy từ cache):", d.shortest("practice", "coding"), "| cache =", d.cache)

shortest('coding', 'practice') = 3 (kỳ vọng 3)
shortest('makes', 'coding') = 1 (kỳ vọng 1)
hỏi lại (lần 2, lấy từ cache): 3 | cache = {('coding', 'practice'): 3, ('coding', 'makes'): 1}


**Độ phức tạp Cách 5:** khởi tạo `O(n)`; truy vấn lần đầu `O(a + b)`, các lần lặp lại **`O(1)`**; **Space `O(n + số cặp đã hỏi)`**.

> Cách 3 là cách nộp bài chuẩn. Cách 4 và 5 là cải tiến tùy dạng truy vấn. Cách 1 và 2 là chuẩn đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 3 (hai con trỏ) trên truy vấn `shortest("makes", "coding")` với `positions` đã dựng, và trên một mảng dài hơn để thấy con trỏ nhảy.

In [16]:
def trace_query(words, word1, word2):
    positions = defaultdict(list)
    for i, w in enumerate(words):
        positions[w].append(i)
    a, b = positions[word1], positions[word2]
    print(f"words={words}")
    print(f"positions[{word1!r}] = {a}, positions[{word2!r}] = {b}")
    i = j = 0
    best = inf
    while i < len(a) and j < len(b):
        diff = abs(a[i] - b[j])
        best = min(best, diff)
        move = "i += 1 (a[i] nhỏ hơn hoặc bằng)" if a[i] <= b[j] else "j += 1 (b[j] nhỏ hơn)"
        print(f"  i={i}, j={j}: a[i]={a[i]}, b[j]={b[j]}, |hiệu|={diff}, best={best} -> {move}")
        if a[i] <= b[j]:
            i += 1
        else:
            j += 1
    print("  => đáp án:", best)
    return best


assert trace_query(WORDS, "makes", "coding") == 1
print()
assert trace_query(["a", "b", "x", "a", "x", "x", "b", "a"], "a", "b") == 1

words=['practice', 'makes', 'perfect', 'coding', 'makes']
positions['makes'] = [1, 4], positions['coding'] = [3]
  i=0, j=0: a[i]=1, b[j]=3, |hiệu|=2, best=2 -> i += 1 (a[i] nhỏ hơn hoặc bằng)
  i=1, j=0: a[i]=4, b[j]=3, |hiệu|=1, best=1 -> j += 1 (b[j] nhỏ hơn)
  => đáp án: 1

words=['a', 'b', 'x', 'a', 'x', 'x', 'b', 'a']
positions['a'] = [0, 3, 7], positions['b'] = [1, 6]
  i=0, j=0: a[i]=0, b[j]=1, |hiệu|=1, best=1 -> i += 1 (a[i] nhỏ hơn hoặc bằng)
  i=1, j=0: a[i]=3, b[j]=1, |hiệu|=2, best=1 -> j += 1 (b[j] nhỏ hơn)
  i=1, j=1: a[i]=3, b[j]=6, |hiệu|=3, best=1 -> i += 1 (a[i] nhỏ hơn hoặc bằng)
  i=2, j=1: a[i]=7, b[j]=6, |hiệu|=1, best=1 -> j += 1 (b[j] nhỏ hơn)
  => đáp án: 1


Bảng tóm tắt (`a = [1, 4]` cho `"makes"`, `b = [3]` cho `"coding"`):

| `i` | `j` | `a[i]` | `b[j]` | `|hiệu|` | `best` | con trỏ tăng |
|---|---|---|---|---|---|---|
| 0 | 0 | 1 | 3 | 2 | 2 | `i` (vì `1 ≤ 3`) |
| 1 | 0 | 4 | 3 | 1 | 1 | `j` (vì `4 > 3`); `j` hết nên dừng |

Đáp án `1`, đúng với đề.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (danh sách chỉ số + hai con trỏ). Đúng tên lớp `WordDistance`, đúng chữ ký `__init__(self, wordsDict)` và `shortest(self, word1, word2)`.

In [17]:
from collections import defaultdict
from math import inf
from typing import List


class WordDistance:
    def __init__(self, wordsDict: List[str]):
        self.d = defaultdict(list)                  # thuộc tính của ĐỐI TƯỢNG (mỗi đối tượng một bản)
        for i, w in enumerate(wordsDict):
            self.d[w].append(i)                     # chỉ số của mỗi từ, tăng dần

    def shortest(self, word1: str, word2: str) -> int:
        a, b = self.d[word1], self.d[word2]
        ans = inf
        i = j = 0
        while i < len(a) and j < len(b):
            ans = min(ans, abs(a[i] - b[j]))
            if a[i] <= b[j]:                        # tiến con trỏ của chỉ số nhỏ hơn
                i += 1
            else:
                j += 1
        return ans


out = run_design(WordDistance, ["WordDistance", "shortest", "shortest"],
                 [[WORDS], ["coding", "practice"], ["makes", "coding"]])
assert out == [None, 3, 1], out
print("PASS ví dụ đề cho class WordDistance:", out)

PASS ví dụ đề cho class WordDistance: [None, 3, 1]


## 10. Vì sao thuật toán đúng

- **Danh sách chỉ số tăng dần:** ta duyệt `i` từ `0` đến `n − 1` và `append(i)`, nên mỗi `d[w]` tăng thật sự.
- **Đáp án cần tìm:** `min |x − y|` với `x ∈ a`, `y ∈ b` (mọi cặp lần xuất hiện của hai từ). `word1 ≠ word2` nên `a` và `b` rời nhau (không có chỉ số nào thuộc cả hai).
- **Hai con trỏ không bỏ sót cặp tối ưu:** bất biến: mọi cặp `(a[i'], b[j'])` có `i' < i` hoặc `j' < j` đã được xét, hoặc không tốt hơn một cặp đã xét. Khi `a[i] ≤ b[j]` và ta tăng `i`: (1) mọi cặp `(a[i], b[j'])` với `j' ≥ j` có khoảng cách `b[j'] − a[i] ≥ b[j] − a[i]`, đã ghi nhận ở bước này; (2) mọi cặp `(a[i], b[j''])` với `j'' < j`: lúc `j` bị tăng vượt `j''`, ta có `a[i*] > b[j'']` với `i* ≤ i` đang xét, nên khoảng cách `a[i] − b[j''] ≥ a[i*] − b[j'']` đã được ghi nhận. Vậy bỏ `a[i]` không làm mất cặp tối ưu. Trường hợp `a[i] > b[j]` đối xứng.
- **Cách 4:** với mỗi `x` trong danh sách ngắn, phần tử gần nhất trong danh sách dài là một trong hai lân cận của vị trí chèn; lấy cả hai nên không bỏ sót.
- **Cách 5:** khoảng cách đối xứng nên khóa không phụ thuộc thứ tự là hợp lệ; kết quả không đổi theo thời gian vì `wordsDict` không đổi sau khởi tạo.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(wordsDict)`, `Q` là số truy vấn, `a`, `b` là số lần xuất hiện của hai từ trong một truy vấn (`a + b ≤ n`).

- **Cách 1 (quét cả mảng mỗi truy vấn):** khởi tạo `O(1)`; mỗi truy vấn `O(n)`; tổng `O(Q·n)`. Space `O(1)` thêm.
- **Cách 2 (mọi cặp chỉ số):** khởi tạo `O(n)`; mỗi truy vấn `O(a·b)`. Space `O(n)`.
- **Cách 3 (hai con trỏ):** khởi tạo `O(n)`; mỗi truy vấn `O(a + b)`. Space `O(n)`.
- **Cách 4 (nhị phân):** khởi tạo `O(n)`; mỗi truy vấn `O(min(a, b)·log max(a, b))`. Space `O(n)`.
- **Cách 5 (cache):** như Cách 3 nhưng truy vấn lặp lại `O(1)`. Space `O(n + số cặp)`.
- Với `n = 3·10⁴`, `Q = 5000`: Cách 1 tối đa `1.5·10⁸` bước; Cách 3 tối đa `Q·n` ở trường hợp xấu nhưng với dữ liệu thông thường (nhiều từ khác nhau) mỗi truy vấn chỉ cỡ vài trăm bước.

## 12. Bộ test

Ta gom các lớp vào từ điển `impls` (kể cả `WordDistance`). `oracle` duyệt **mọi cặp chỉ số** `(i, j)` trên mảng gốc (không dùng `dict` hay con trỏ). `run_design` mô phỏng đúng định dạng test của bài thiết kế.

In [18]:
impls = {
    "scan": WordDistanceScan,
    "pairs": WordDistancePairs,
    "two_pointers": WordDistanceTwoPointers,
    "bisect": WordDistanceBisect,
    "cached": WordDistanceCached,
    "submit": WordDistance,
}


def oracle(words, w1, w2):
    best = None
    for i in range(len(words)):
        if words[i] != w1:
            continue
        for j in range(len(words)):
            if words[j] == w2:
                d = abs(i - j)
                best = d if best is None else min(best, d)
    return best


def run_queries(cls, words, queries):
    obj = cls(list(words))
    return [obj.shortest(w1, w2) for w1, w2 in queries]


# ví dụ của đề theo đúng định dạng ops / args
ops = ["WordDistance", "shortest", "shortest"]
args = [[WORDS], ["coding", "practice"], ["makes", "coding"]]
for name, cls in impls.items():
    assert run_design(cls, [cls.__name__] + ops[1:], args) == [None, 3, 1], name
for words, queries in examples:
    for w1, w2, expected in queries:
        assert oracle(words, w1, w2) == expected
print("PASS ví dụ của đề (mọi lớp, qua run_design; oracle độc lập)")

PASS ví dụ của đề (mọi lớp, qua run_design; oracle độc lập)


### 12.1. Các trường hợp biên

In [19]:
def check_all(words, queries, label):
    expected = [oracle(words, w1, w2) for w1, w2 in queries]
    for name, cls in impls.items():
        got = run_queries(cls, words, queries)
        assert got == expected, (label, name, got, expected)
    print("PASS", label, "->", expected)


check_all(["a", "b"], [("a", "b"), ("b", "a")], "n = 2, hai từ kề nhau, hỏi hai chiều")
check_all(["a", "x", "b"], [("a", "b")], "cách nhau 2")
check_all(["a", "b", "a", "b", "a"], [("a", "b"), ("b", "a")], "xen kẽ: khoảng cách 1")
check_all(["a", "x", "x", "x", "b", "x", "a"], [("a", "b")], "chọn lần xuất hiện tốt nhất của a (chỉ số 6, không phải 0)")
check_all(["b", "x", "x", "a", "b"], [("a", "b")], "lỗi 'chỉ lấy chỉ số đầu tiên' bị bắt")
check_all(["a"] * 5 + ["b"] + ["a"] * 5, [("a", "b")], "từ a phổ biến, từ b hiếm, ở giữa")
check_all(["practice", "makes", "perfect", "coding", "makes"],
          [("coding", "practice"), ("makes", "coding"), ("practice", "makes"), ("perfect", "makes")], "ví dụ của đề + thêm truy vấn")
check_all(list("abcdefghij"), [("a", "j"), ("j", "a"), ("e", "f")], "mọi từ khác nhau")


def check_big(words, queries, label):
    """Dữ liệu lớn: oracle O(a*b) quá chậm nên dùng bản quét O(n) làm chuẩn; bỏ qua bản 'mọi cặp'."""
    expected = run_queries(WordDistanceScan, words, queries)
    for name, cls in impls.items():
        if name == "pairs":
            continue
        assert run_queries(cls, words, queries) == expected, (label, name)
    print("PASS", label, "->", expected)


check_big(["a", "b"] * 15000, [("a", "b")], "n = 3 * 10^4, xen kẽ")
check_big(["a"] * 15000 + ["b"] * 15000, [("a", "b")], "n = 3 * 10^4, hai khối: khoảng cách 1 ở chỗ nối")
check_big(["a"] + ["x"] * 29998 + ["b"], [("a", "b")], "n = 3 * 10^4, hai từ ở hai đầu: khoảng cách n - 1")

PASS n = 2, hai từ kề nhau, hỏi hai chiều -> [1, 1]
PASS cách nhau 2 -> [2]
PASS xen kẽ: khoảng cách 1 -> [1, 1]
PASS chọn lần xuất hiện tốt nhất của a (chỉ số 6, không phải 0) -> [2]
PASS lỗi 'chỉ lấy chỉ số đầu tiên' bị bắt -> [1]
PASS từ a phổ biến, từ b hiếm, ở giữa -> [1]
PASS ví dụ của đề + thêm truy vấn -> [3, 1, 1, 1]
PASS mọi từ khác nhau -> [9, 9, 1]
PASS n = 3 * 10^4, xen kẽ -> [1]
PASS n = 3 * 10^4, hai khối: khoảng cách 1 ở chỗ nối -> [1]
PASS n = 3 * 10^4, hai từ ở hai đầu: khoảng cách n - 1 -> [29999]


### 12.2. Tính chất và phản ví dụ

In [20]:
random.seed(244)

# Tính chất 1: đối xứng shortest(a, b) == shortest(b, a); luôn >= 1; <= n - 1
for _ in range(300):
    n = random.randint(2, 30)
    words = [random.choice("abcd") for _ in range(n)]
    distinct = sorted(set(words))
    if len(distinct) < 2:
        continue
    w1, w2 = random.sample(distinct, 2)
    d = WordDistance(words)
    x, y = d.shortest(w1, w2), d.shortest(w2, w1)
    assert x == y and 1 <= x <= n - 1
print("PASS tính chất: đối xứng và 1 <= khoảng cách <= n - 1")

# Tính chất 2: nhiều đối tượng độc lập không ảnh hưởng nhau
d1 = WordDistance(["a", "b", "x", "x", "x"])
d2 = WordDistance(["b", "x", "x", "x", "a"])
assert d1.shortest("a", "b") == 1 and d2.shortest("a", "b") == 4
assert d1.shortest("a", "b") == 1
print("PASS tính chất: hai đối tượng WordDistance độc lập")

# Tính chất 3: cache không đổi kết quả khi hỏi lặp lại
cached = WordDistanceCached(["a", "x", "b", "a", "b"])
first = [cached.shortest("a", "b"), cached.shortest("b", "a"), cached.shortest("a", "b")]
assert first == [1, 1, 1]

# Phản ví dụ: dùng thuộc tính của LỚP (dùng chung giữa các đối tượng)
class WordDistanceShared:
    pos = defaultdict(list)                            # thuộc tính của lớp: dùng chung!

    def __init__(self, wordsDict):
        for i, w in enumerate(wordsDict):
            self.pos[w].append(i)

    def shortest(self, word1, word2):
        a, b = self.pos[word1], self.pos[word2]
        return min(abs(x - y) for x in a for y in b)

s1 = WordDistanceShared(["a", "b", "x", "x", "x"])
s2 = WordDistanceShared(["b", "x", "x", "x", "a"])
print("đối tượng 1 sau khi tạo đối tượng 2 -> sai:", s1.shortest("a", "b"), "| đúng: 1")
assert s1.shortest("a", "b") != 1 or s2.shortest("a", "b") != 4

PASS tính chất: đối xứng và 1 <= khoảng cách <= n - 1
PASS tính chất: hai đối tượng WordDistance độc lập
đối tượng 1 sau khi tạo đối tượng 2 -> sai: 0 | đúng: 1


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [21]:
random.seed(2026)
for trial in range(800):
    n = random.randint(2, 40)
    alphabet = random.choice(["ab", "abc", "abcdef", [str(i) for i in range(10)]])
    words = [random.choice(alphabet) for _ in range(n)]
    distinct = sorted(set(words))
    if len(distinct) < 2:
        continue
    queries = [tuple(random.sample(distinct, 2)) for _ in range(random.randint(1, 8))]
    expected = [oracle(words, w1, w2) for w1, w2 in queries]
    for name, cls in impls.items():
        assert run_queries(cls, words, queries) == expected, (name, words, queries)

# dữ liệu lớn: n = 3000, 300 truy vấn; so các lớp với nhau (scan là chuẩn O(n) mỗi truy vấn)
vocab = [f"w{i}" for i in range(30)]
words = [random.choice(vocab) for _ in range(3000)]
queries = [tuple(random.sample(vocab, 2)) for _ in range(300)]
base = run_queries(WordDistanceScan, words, queries)
for name, cls in impls.items():
    assert run_queries(cls, words, queries) == base, name
print("PASS 800 mảng ngẫu nhiên khớp oracle độc lập; dữ liệu lớn khớp giữa các lớp")

PASS 800 mảng ngẫu nhiên khớp oracle độc lập; dữ liệu lớn khớp giữa các lớp


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ lưu MỘT chỉ số cho mỗi từ (ghi đè hoặc chỉ lấy đầu tiên)

In [22]:
class SaiMotChiSo:
    def __init__(self, wordsDict):
        self.idx = {}
        for i, w in enumerate(wordsDict):
            if w not in self.idx:
                self.idx[w] = i                    # chỉ nhớ lần xuất hiện ĐẦU TIÊN

    def shortest(self, word1, word2):
        return abs(self.idx[word1] - self.idx[word2])


words = ["b", "x", "x", "a", "b"]
print("sai:", SaiMotChiSo(words).shortest("a", "b"), "| đúng:", WordDistance(words).shortest("a", "b"))

sai: 3 | đúng: 1


**Vì sao sai:** một từ có thể xuất hiện nhiều lần; khoảng cách ngắn nhất có thể dùng **bất kỳ** lần xuất hiện nào. `"b"` xuất hiện ở `0` và `4`; cặp tốt nhất `(3, 4)` cách `1`, nhưng chỉ nhớ chỉ số đầu `0` cho `3`. Cần **danh sách** chỉ số cho mỗi từ.

### Lỗi 2: Tăng sai con trỏ (tăng con trỏ trỏ vào chỉ số lớn hơn)

In [23]:
class SaiConTro:
    def __init__(self, wordsDict):
        self.d = defaultdict(list)
        for i, w in enumerate(wordsDict):
            self.d[w].append(i)

    def shortest(self, word1, word2):
        a, b = self.d[word1], self.d[word2]
        ans, i, j = inf, 0, 0
        while i < len(a) and j < len(b):
            ans = min(ans, abs(a[i] - b[j]))
            if a[i] <= b[j]:
                j += 1                              # sai: phải tăng i (chỉ số nhỏ hơn)
            else:
                i += 1
        return ans


words = ["a", "x", "b", "x", "x", "x", "a", "b", "x", "a"]
print("sai:", SaiConTro(words).shortest("a", "b"), "| đúng:", WordDistance(words).shortest("a", "b"))

sai: 2 | đúng: 1


**Vì sao sai:** phải **tăng con trỏ của chỉ số nhỏ hơn**: chỉ số nhỏ hơn không thể gần hơn với các chỉ số sau của list kia (còn lớn hơn). Tăng con trỏ của chỉ số lớn hơn bỏ qua các cặp có thể tốt hơn.

### Lỗi 3: Quên `abs` (khoảng cách âm)

In [24]:
class SaiQuenAbs:
    def __init__(self, wordsDict):
        self.d = defaultdict(list)
        for i, w in enumerate(wordsDict):
            self.d[w].append(i)

    def shortest(self, word1, word2):
        a, b = self.d[word1], self.d[word2]
        ans, i, j = inf, 0, 0
        while i < len(a) and j < len(b):
            ans = min(ans, a[i] - b[j])             # sai: có thể âm
            if a[i] <= b[j]:
                i += 1
            else:
                j += 1
        return ans


print("sai:", SaiQuenAbs(WORDS).shortest("coding", "makes"), "| đúng:", WordDistance(WORDS).shortest("coding", "makes"))

sai: -1 | đúng: 1


**Vì sao sai:** khoảng cách là `|i − j|` (luôn không âm). Khi `word1` đứng trước `word2` thì `a[i] − b[j] < 0`. Phải dùng `abs`, nếu không `min` chọn số âm có độ lớn lớn nhất.

### Lỗi 4: Dùng thuộc tính của LỚP thay vì của ĐỐI TƯỢNG

In [25]:
class SaiThuocTinhLop:
    d = defaultdict(list)                           # sai: khai báo ở cấp lớp -> dùng chung mọi đối tượng

    def __init__(self, wordsDict):
        for i, w in enumerate(wordsDict):
            self.d[w].append(i)

    def shortest(self, word1, word2):
        a, b = self.d[word1], self.d[word2]
        return min(abs(x - y) for x in a for y in b)


first = SaiThuocTinhLop(["a", "b", "x", "x", "x"])
second = SaiThuocTinhLop(["b", "x", "x", "x", "a"])
print("đối tượng 1 -> sai:", first.shortest("a", "b"), "| đúng: 1")

đối tượng 1 -> sai: 0 | đúng: 1


**Vì sao sai:** `d = defaultdict(list)` viết ở cấp lớp tạo **một** từ điển dùng chung. Mỗi lần tạo đối tượng mới, chỉ số được **thêm vào cùng một từ điển**, làm dữ liệu các đối tượng lẫn nhau. Hệ thống chấm của LeetCode tạo nhiều đối tượng qua các test, nên lỗi này cho đáp án sai. Khởi tạo dữ liệu trong `__init__` bằng `self.d = ...`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thiết kế "khởi tạo một lần, **truy vấn nhiều lần**" → **tiền xử lý** ở `__init__` để mỗi truy vấn rẻ.
- "Vị trí xuất hiện của từng giá trị" → `defaultdict(list)` với `enumerate`; list tự **đã sắp**.
- "Khoảng cách nhỏ nhất giữa hai list đã sắp" → **hai con trỏ** (tăng con trỏ của số nhỏ hơn) hoặc **nhị phân** khi một list ngắn hơn nhiều.
- Dữ liệu và trạng thái của đối tượng phải ở `self.` (không phải thuộc tính của lớp, cũng không phải biến toàn cục).
- Khi hàm đối xứng và hỏi lặp lại: **cache** theo khóa không phụ thuộc thứ tự.
- Test bài thiết kế: viết `run_design(cls, ops, args)` để mô phỏng định dạng LeetCode, và oracle duyệt mọi cặp chỉ số.

## 15. Biến thể của bài

1. **Shortest Word Distance I (LeetCode 243):** chỉ một truy vấn, quét một lần `O(n)`.
2. **Shortest Word Distance III (LeetCode 245):** `word1` có thể bằng `word2` (bài kế tiếp).
3. **Khoảng cách **dài nhất** giữa hai từ.**
4. **Truy vấn cho nhiều từ:** khoảng cách ngắn nhất giữa một từ và **bất kỳ** từ trong một tập.
5. **Dữ liệu động:** thêm từ vào cuối mảng (streaming): danh sách chỉ số vẫn tăng.

In [26]:
def shortest_distance_one_query(words, word1, word2):
    """Biến thể 1: một truy vấn, quét một lần."""
    i1 = i2 = -1
    best = len(words)
    for k, w in enumerate(words):
        if w == word1:
            i1 = k
        elif w == word2:
            i2 = k
        if i1 != -1 and i2 != -1:
            best = min(best, abs(i1 - i2))
    return best


class WordDistanceExtended(WordDistance):
    def longest(self, word1, word2):
        """Biến thể 3: khoảng cách lớn nhất = max(|đầu của một - cuối của kia|)."""
        a, b = self.d[word1], self.d[word2]
        return max(abs(a[0] - b[-1]), abs(a[-1] - b[0]))

    def shortest_to_any(self, word, others):
        """Biến thể 4: khoảng cách ngắn nhất từ word tới bất kỳ từ nào trong others."""
        return min(self.shortest(word, o) for o in others)

    def add(self, word):
        """Biến thể 5: thêm một từ vào cuối mảng (chỉ số mới lớn hơn mọi chỉ số cũ)."""
        self.n = getattr(self, "n", sum(len(v) for v in self.d.values()))
        self.d[word].append(self.n)
        self.n += 1


ext = WordDistanceExtended(WORDS)
assert shortest_distance_one_query(WORDS, "makes", "coding") == 1
assert ext.longest("makes", "practice") == 4 and ext.longest("coding", "practice") == 3
assert ext.shortest_to_any("coding", ["practice", "perfect"]) == 1
ext.add("coding")                                   # mảng thành [..., makes, coding]
assert ext.shortest("coding", "makes") == 1 and ext.d["coding"] == [3, 5]
random.seed(15)
for _ in range(200):
    n = random.randint(2, 20)
    words = [random.choice("abc") for _ in range(n)]
    if len(set(words)) < 2:
        continue
    w1, w2 = random.sample(sorted(set(words)), 2)
    assert shortest_distance_one_query(words, w1, w2) == oracle(words, w1, w2) == WordDistanceExtended(words).shortest(w1, w2)
print("PASS biến thể: một truy vấn, khoảng cách dài nhất, tập từ, thêm động")

PASS biến thể: một truy vấn, khoảng cách dài nhất, tập từ, thêm động


## 16. So sánh các cách làm

| Cách | Ý tưởng | Khởi tạo | Mỗi truy vấn | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét cả mảng | không tiền xử lý | `O(1)` | `O(n)` | ít truy vấn; oracle |
| 2. Danh sách chỉ số + mọi cặp | `min |x − y|` trên mọi cặp | `O(n)` | `O(a·b)` | đơn giản, từ hiếm |
| 3. Danh sách chỉ số + hai con trỏ | trộn hai list đã sắp | `O(n)` | `O(a + b)` | **cách nộp bài** |
| 4. Danh sách chỉ số + nhị phân | `bisect_left` trên list dài | `O(n)` | `O(min·log max)` | một từ hiếm, một từ phổ biến |
| 5. Hai con trỏ + cache | lưu kết quả theo cặp | `O(n)` | `O(a + b)` lần đầu, `O(1)` lặp lại | truy vấn lặp lại |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `abs(2 - 7)`, `dict(defaultdict(list, {"a": [0]}))`, `merge_min_gap([0, 10], [4])`, `WordDistance(["a", "b"]).shortest("a", "b")`, `WordDistance(["a", "x", "b"]).shortest("b", "a")`.

**Bài 2 (tính tay).** Với `words = ["a", "b", "x", "a", "x", "b", "a"]`: ghi `positions["a"]`, `positions["b"]` và chạy hai con trỏ cho `shortest("a", "b")`, liệt kê từng bước.

**Bài 3 (điền TODO).** Hoàn thành `class WordDistanceTodo` theo Cách 3.

In [27]:
class WordDistanceTodo:
    def __init__(self, wordsDict):
        # TODO 1: self.d = defaultdict(list); duyệt i, w bằng enumerate và self.d[w].append(i)
        pass

    def shortest(self, word1, word2):
        # TODO 2: a, b = self.d[word1], self.d[word2]; ans = inf; i = j = 0
        # TODO 3: trong khi i < len(a) và j < len(b): cập nhật ans = min(ans, abs(a[i] - b[j])),
        #         rồi tăng con trỏ của chỉ số nhỏ hơn
        # TODO 4: trả về ans
        pass

# KHÔNG tạo đối tượng / gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `class WordDistanceBisect2` theo Cách 4 (nhị phân trên list dài hơn). Kiểm tra bằng ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `class WordDistanceK` có thêm phương thức `kth_closest(word1, word2, k)` trả về khoảng cách ngắn nhất thứ `k` (các khoảng cách sắp tăng, tính mọi cặp). (b) Viết `class WordDistanceLongest` có `longest(word1, word2)` trả khoảng cách **lớn nhất**.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `abs` là giá trị tuyệt đối; khoảng cách giữa chỉ số `0` và `1` là `1`; `shortest` đối xứng. Bài 2: viết các chỉ số của từng từ.
- **Mức 2 (vừa):** Bài 1: `5`, `{'a': [0]}`, `4` (`|0−4| = 4`, `|10−4| = 6`), `1`, `2`. Bài 2: `positions["a"] = [0, 3, 6]`, `positions["b"] = [1, 5]`: bước `(0,1)`: 1; vì `0 ≤ 1` tăng `i`; `(3,1)`: 2 vì `3 > 1` tăng `j`; `(3,5)`: 2 vì `3 ≤ 5` tăng `i`; `(6,5)`: 1; `6 > 5` tăng `j` hết; đáp án `1`. Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `short, long_ = sorted((a, b), key=len)`; với mỗi `x` trong `short` dùng `bisect_left(long_, x)` và xét `long_[k]`, `long_[k - 1]`. Bài 5(a): `sorted(abs(x - y) for x in a for y in b)[k - 1]`. Bài 5(b): `max(abs(a[0] - b[-1]), abs(a[-1] - b[0]))` vì các chỉ số đã sắp.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert abs(2 - 7) == 5
assert dict(defaultdict(list, {"a": [0]})) == {"a": [0]}
assert merge_min_gap([0, 10], [4]) == 4
assert WordDistance(["a", "b"]).shortest("a", "b") == 1
assert WordDistance(["a", "x", "b"]).shortest("b", "a") == 2

# Bài 2
w2 = ["a", "b", "x", "a", "x", "b", "a"]
d2 = WordDistance(w2)
assert d2.d["a"] == [0, 3, 6] and d2.d["b"] == [1, 5]
assert d2.shortest("a", "b") == 1 == oracle(w2, "a", "b")

# Bài 3
class WordDistanceTodoAns:
    def __init__(self, wordsDict):
        self.d = defaultdict(list)
        for i, w in enumerate(wordsDict):
            self.d[w].append(i)

    def shortest(self, word1, word2):
        a, b = self.d[word1], self.d[word2]
        ans, i, j = inf, 0, 0
        while i < len(a) and j < len(b):
            ans = min(ans, abs(a[i] - b[j]))
            if a[i] <= b[j]:
                i += 1
            else:
                j += 1
        return ans

assert run_design(WordDistanceTodoAns, ["WordDistanceTodoAns", "shortest", "shortest"],
                  [[WORDS], ["coding", "practice"], ["makes", "coding"]]) == [None, 3, 1]

# Bài 4
class WordDistanceBisect2:
    def __init__(self, wordsDict):
        self.d = defaultdict(list)
        for i, w in enumerate(wordsDict):
            self.d[w].append(i)

    def shortest(self, word1, word2):
        short, long_ = sorted((self.d[word1], self.d[word2]), key=len)
        best = inf
        for x in short:
            k = bisect_left(long_, x)
            if k < len(long_):
                best = min(best, long_[k] - x)
            if k:
                best = min(best, x - long_[k - 1])
        return best

assert run_design(WordDistanceBisect2, ["WordDistanceBisect2", "shortest", "shortest"],
                  [[WORDS], ["coding", "practice"], ["makes", "coding"]]) == [None, 3, 1]

# Bài 5(a), 5(b)
class WordDistanceK(WordDistance):
    def kth_closest(self, word1, word2, k):
        a, b = self.d[word1], self.d[word2]
        return sorted(abs(x - y) for x in a for y in b)[k - 1]

class WordDistanceLongest(WordDistance):
    def longest(self, word1, word2):
        a, b = self.d[word1], self.d[word2]
        return max(abs(a[0] - b[-1]), abs(a[-1] - b[0]))

wk = WordDistanceK(WORDS)
assert wk.kth_closest("makes", "coding", 1) == 1 and wk.kth_closest("makes", "coding", 2) == 2
wl = WordDistanceLongest(WORDS)
assert wl.longest("makes", "practice") == 4
random.seed(19)
for _ in range(200):
    n = random.randint(2, 15)
    words = [random.choice("abc") for _ in range(n)]
    if len(set(words)) < 2:
        continue
    x, y = random.sample(sorted(set(words)), 2)
    brute_long = max(abs(i - j) for i in range(n) for j in range(n) if words[i] == x and words[j] == y)
    assert WordDistanceLongest(words).longest(x, y) == brute_long
    assert WordDistanceBisect2(words).shortest(x, y) == oracle(words, x, y)
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (danh sách chỉ số của từng từ + hai con trỏ)

```python
class WordDistance:
    def __init__(self, wordsDict: List[str]):
        self.d = defaultdict(list)
        for i, w in enumerate(wordsDict):
            self.d[w].append(i)

    def shortest(self, word1: str, word2: str) -> int:
        a, b = self.d[word1], self.d[word2]
        ans = inf
        i = j = 0
        while i < len(a) and j < len(b):
            ans = min(ans, abs(a[i] - b[j]))
            if a[i] <= b[j]:
                i += 1
            else:
                j += 1
        return ans
```

**Checklist**

- [ ] Tiền xử lý ở `__init__`: mỗi từ có **danh sách** chỉ số (không chỉ một chỉ số)?
- [ ] Dữ liệu lưu bằng `self.xxx` (thuộc tính **đối tượng**), không dùng thuộc tính lớp hay biến toàn cục?
- [ ] Hai con trỏ: tăng con trỏ của **chỉ số nhỏ hơn**; cập nhật `ans` bằng `abs`?
- [ ] Đáp án khởi tạo bằng `inf` (hoặc `n`) và luôn được cập nhật (vì hai từ đều có trong mảng)?
- [ ] Đã thử: `n = 2`, từ xuất hiện nhiều lần, hỏi hai chiều `(a, b)` và `(b, a)`, nhiều đối tượng độc lập?
- [ ] Đã đối chiếu với oracle duyệt mọi cặp chỉ số trên mảng ngẫu nhiên, mô phỏng đúng định dạng `ops`/`args`?
- [ ] Skeleton đúng `class WordDistance` / `__init__(self, wordsDict: List[str])` / `shortest(self, word1: str, word2: str) -> int`?